In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv("../Data/processed/clean_house_data.csv")

In [3]:
df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,bhk,price_per_sqft,sqft_per_bhk
0,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,1875.0,3.0,1.0,167.0,3,8906.666667,625.000000
1,Built-up Area,Ready To Move,1st Phase JP Nagar,5 Bedroom,1500.0,5.0,2.0,85.0,5,5666.666667,300.000000
2,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,2065.0,4.0,1.0,210.0,3,10169.491525,688.333333
3,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,2059.0,3.0,2.0,225.0,3,10927.634774,686.333333
4,Super built-up Area,Ready To Move,1st Phase JP Nagar,2 BHK,1394.0,2.0,1.0,100.0,2,7173.601148,697.000000


In [4]:
bins = [0, 50, 100, 150, 200, 300, 500, 2500]

df["price_class"] = pd.cut(
    df["price"],
    bins=bins,
    labels=False
)

In [5]:
df["price_class"].value_counts().sort_index()

price_class
0    1865
1    3286
2    1140
3     406
4     361
5     194
6      44
Name: count, dtype: int64

In [6]:
df_model = df.drop(
    columns=["price", "price_per_sqft"]
)

In [7]:
X = df_model.drop("price_class", axis=1)

y = df_model["price_class"]

In [8]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   #the train and test sets might have very different class distributions. With it, both sets maintain approximately the same proportion of each class.
)

In [10]:
print(y_train.value_counts().sort_index())

price_class
0    1492
1    2628
2     912
3     325
4     289
5     155
6      35
Name: count, dtype: int64


# SMOTE

In [11]:
# Synthetic Minority Over-sampling Technique

# 1. Split into train and test.
# 2. Apply SMOTE only on the training set.
# 3. Train the model on the balanced training data.
# 4. Evaluate on the untouched test set.



In [12]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [13]:
print(y_train_smote.value_counts().sort_index())


price_class
0    2628
1    2628
2    2628
3    2628
4    2628
5    2628
6    2628
Name: count, dtype: int64


## training

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(
    X_train_smote,
    y_train_smote
)

RandomForestClassifier(random_state=42)

In [15]:
y_pred = rf.predict(X_test)

## evaluate

In [16]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.7575342465753425

In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.81      0.84      0.82       373
           1       0.83      0.81      0.82       658
           2       0.65      0.68      0.67       228
           3       0.41      0.42      0.42        81
           4       0.60      0.57      0.59        72
           5       0.68      0.67      0.68        39
           6       0.80      0.44      0.57         9

    accuracy                           0.76      1460
   macro avg       0.68      0.63      0.65      1460
weighted avg       0.76      0.76      0.76      1460



# light gbm
best classifier when check w automl

In [18]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(random_state=42)

lgbm.fit(
    X_train_smote,
    y_train_smote
)

y_pred = lgbm.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012869 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1504
[LightGBM] [Info] Number of data points in the train set: 18396, number of used features: 240
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start training from score -1.945910
Accuracy: 0.7664383561643836
              precision    recall  f1-score   support

           0       0.83      0.84      0.84       373
           1       0.83      0.83      0.83       

***

## Conclusion

- Converted house prices into discrete price ranges.
- Observed class imbalance.
- Applied SMOTE on the training data only.
- Balanced all classes successfully.
- Trained Random Forest and LightGBM classifiers.
- LightGBM achieved the highest accuracy (~76.6%).
- SMOTE improved classification performance compared to the original imbalanced dataset.
- Regression remains the preferred approach for exact house price prediction, while classification is suitable for predicting approximate price bands.